# Novelty Engine

In [1]:
import importlib
import sys
import os

# Add the project root to Python path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Set dummy OpenAI API key to prevent RAGAS from requiring real OpenAI credentials
os.environ["OPENAI_API_KEY"] = "dummy-key-for-ragas"

# Import the novelty module
from sspbench.novelty import *
from sspbench.novelty.main import run_novelty_engine
from sspbench.novelty.ragas_utils import SentenceTransformerEmbeddings, is_ragas_available

os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "7")


INFO 02-05 17:15:59 [__init__.py:216] Automatically detected platform cuda.


/home/local/QCRI/fdeniz/anaconda3/envs/autobencher/lib/python3.10/site-packages/flaml/__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


In [2]:
# Reload modules for development
importlib.reload(sys.modules['sspbench.novelty.config'])
importlib.reload(sys.modules['sspbench.novelty.wiki_utils'])
importlib.reload(sys.modules['sspbench.novelty.llm_utils'])
importlib.reload(sys.modules['sspbench.novelty.ragas_utils'])
importlib.reload(sys.modules['sspbench.novelty.evaluation'])
importlib.reload(sys.modules['sspbench.novelty.variations'])
importlib.reload(sys.modules['sspbench.novelty.core'])
importlib.reload(sys.modules['sspbench.novelty.main'])
importlib.reload(sys.modules['sspbench.novelty'])

from sspbench.novelty import *
from sspbench.novelty.main import run_novelty_engine

print("Modules reloaded successfully!")

Modules reloaded successfully!


## Configuration

Set up your models and parameters here.

In [3]:
agent_config = {
    "type": "openai",
    "model": "gpt-4.1-mini-aixamine",
    "api_url": "https://qcri-oai-aixamine-01.openai.azure.com/",
    "api_token": "549HzN76L1k2WMzrqQoN26dJ2TItgVpsz21f9yjk342PWwj9RzrmJQQJ99BIACYeBjFXJ3w3AAABACOGAkdr",
    "api_version": "2024-12-01-preview"
}

test_config = {
    "type": "huggingface",
    "model": "/home/local/QCRI/fdeniz/projects/aiXamine/airflow-tasks/models/google_gemma-2-2b-it"
}

eval_config = {
    "type": "openai",
    "model": "gpt-oss",
    "api_url": "http://10.4.8.217:8000/v1",
    "api_token": "abc123",
    "api_version": "2024-12-01-preview"
}

embedding_model = SentenceTransformerEmbeddings("all-MiniLM-L6-v2") if is_ragas_available() else None

models_already_exist = 'models_created' in globals() and models_created
if not models_already_exist:
    models_created = {}
    failed_models = {}

    for name, config in [("agent", agent_config), ("test", test_config), ("eval", eval_config)]:
        try:
            model = create_model_from_config(config)
            models_created[name] = model
            print(f"\033[92m✓ {name}_model created successfully\033[0m")
            test_response = model.generate("Hello")
            print(f"\033[92m✓ {name}_model.generate() works: {test_response[:50]}...\033[0m")
        except Exception as e:
            print(f"✗ Failed to create {name}_model: {e}")
            failed_models[name] = str(e)
            models_created[name] = None
    
    print("\n✓ Models initialized for the first time!")
else:
    print("✓ Using previously created models (skipping initialization)")


agent_model = models_created.get("agent")
test_model = models_created.get("test") 
eval_model = models_created.get("eval")

# Parameters
theme = "general knowledge"
max_iterations = 3
acc_target = "0.1--0.4"
engine = "novelty"

print("\nModels configured:")
print(f"Agent: {agent_config.get('model', 'Unknown')} {'(failed)' if 'agent' in failed_models else ''}")
print(f"Test: {test_config.get('model', 'Unknown')} {'(failed)' if 'test' in failed_models else ''}")
print(f"Eval: {eval_config.get('model', 'Unknown')} {'(failed)' if 'eval' in failed_models else ''}")
print(f"Theme: {theme}")
print(f"Max iterations: {max_iterations}")

if failed_models:
    print(f"\n⚠️  {len(failed_models)} model(s) failed to load. You can still run the engine with working models.")
    print("Failed models:", list(failed_models.keys()))
else:
    print("\n✓ All models loaded successfully!")

print("\nNote: Models are cached to avoid reloading. Use clear_model_cache() to clear cache if needed.")

Loading new model: gpt-4.1-mini-aixamine
[Warning] Generation config not defined: {}, using defaults.
Generated Model: AzureOpenaiLLM Fail on empty response: False
✓ agent_model created successfully
Judge batch API call failed: BadRequestError("Error code: 400 - {'error': {'code': 'OperationNotSupported', 'message': 'The completion operation does not work with the specified model, gpt-4.1-mini. Please choose different model and try again. You can learn more about which models can be used with each operation here: https://go.microsoft.com/fwlink/?linkid=2197993.'}}"). Falling back to chat completions...
✓ agent_model.generate() works: ['Hello! How can I assist you today?']...
Loading new model: /home/local/QCRI/fdeniz/projects/aiXamine/airflow-tasks/models/google_gemma-2-2b-it
[Warning] Generation config not defined: {}, using defaults.
[Info] Trying attention backend: XFORMERS
[Try] Loading model with config: dtype=auto | quant=none | backend=XFORMERS | tp=1 | max_len=model-default
INF

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-05 17:16:22 [__init__.py:742] Resolved architecture: Gemma2ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 02-05 17:16:22 [__init__.py:2716] Your device 'Tesla V100-SXM2-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float32 for compatibility.
INFO 02-05 17:16:22 [__init__.py:2761] Upcasting torch.bfloat16 to torch.float32.
INFO 02-05 17:16:22 [__init__.py:1815] Using max model len 8192
WARNING 02-05 17:16:22 [arg_utils.py:1801] VLLM_ATTENTION_BACKEND=XFORMERS is not supported by the V1 Engine. Falling back to V0. We recommend to remove VLLM_ATTENTION_BACKEND=XFORMERS from your config in favor of the V1 Engine.
WARNING 02-05 17:16:22 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 02-05 17:16:22 [__init__.py:3400] Cudagraph is disabled under eager mode
INFO 02-05 17:16:22 [llm_engine.py:221] Initializing a V0 LLM engine (v0.10.2) with config: model='/home/local/QCRI/fdeniz/projects/aiXamine/airflow-tasks/models/google_gemma-2-2b-it', spe

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[Success] Loaded with dtype=auto | quant=none | backend=XFORMERS | tp=1 | max_len=model-default
Generated Model: HuggingFaceLLM Fail on empty response: True
✓ test_model created successfully
Failed to apply chat template with exception System role not supported


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

✓ test_model.generate() works: ['Hello! 👋  How can I help you today? 😊']...
Loading new model: gpt-oss
[Warning] Generation config not defined: {}, using defaults.
Discovered supported parameters: ['max_tokens', 'temperature', 'top_p', 'presence_penalty']
Generated Model: OpenaiLLM Fail on empty response: False
✓ eval_model created successfully
Content in batch was blocked by API model, trying chat inference...
✓ eval_model.generate() works: ['Hello! How can I assist you today?']...

✓ Models initialized for the first time!

Models configured:
Agent: gpt-4.1-mini-aixamine 
Test: /home/local/QCRI/fdeniz/projects/aiXamine/airflow-tasks/models/google_gemma-2-2b-it 
Eval: gpt-oss 
Theme: general knowledge
Max iterations: 3

✓ All models loaded successfully!

Note: Models are cached to avoid reloading. Use clear_model_cache() to clear cache if needed.


## Run the Novelty Engine

Execute the main novelty engine pipeline.

In [ ]:
# Run the novelty engine
history = run_novelty_engine(
    agent_model=agent_model,
    test_model=test_model,
    eval_model=eval_model,
    theme=theme,
    max_iterations=max_iterations,
    acc_target=acc_target,
    engine=engine,
    use_ragas=True,
    embedding_model=embedding_model
)

print("\nNovelty Engine completed!")
print(f"Total iterations: {len(history)}")
print(f"Results saved in data/{engine}/ directory (relative to project root)")

🔬 RAGAS integration enabled for question generation

=== Iteration 1 ===
Iteration 1, SUMMARY: Initial iteration...
Found entity World History


Applying KeyphrasesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

...

Applying NERExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

## Results Analysis

Analyze the results from the runs.

In [ ]:
# Analyze results
if 'history' in locals():
    print("Novelty Engine Results:")
    for i, iteration_results in enumerate(history):
        if iteration_results:
            summary = get_summary_of_results(iteration_results, gold_key='gold_answer', verbose=False)
            acc_lst = get_acc_lst(iteration_results)
            avg_acc = sum(acc_lst) / len(acc_lst) if acc_lst else 0
            print(f"Iteration {i+1}: Avg Accuracy = {avg_acc:.3f}")
            print(f"Summary: {summary[:100]}...")
        print()

# Load and display generated questions
import json
import os

data_dir = os.path.join('..', 'data', engine)
all_questions = []
for iter_num in range(1, max_iterations + 1):
    questions_file = os.path.join(data_dir, f"{theme.replace(' ', '_')}_iter_{iter_num}.KI_questions.json")
    if os.path.exists(questions_file):
        with open(questions_file, 'r') as f:
            questions = json.load(f)
        print(f"\nIteration {iter_num} Questions ({len(questions)} total):")
        for j, q in enumerate(questions, 1):
            print(f"{j}. {q['question']}")
            print(f"   Answer: {q['gold_answer']}")
            if 'faithfulness' in q:
                print(f"   Faithfulness: {q['faithfulness']}")
            print()
            all_questions.append(q)
    else:
        print(f"No questions file found for iteration {iter_num}")

# Save all questions to a text file
if all_questions:
    questions_txt_path = os.path.join(data_dir, f"{theme.replace(' ', '_')}_all_questions.txt")
    with open(questions_txt_path, 'w') as f:
        f.write(f"Generated Questions for Theme: {theme}\n")
        f.write(f"Total Questions: {len(all_questions)}\n\n")
        for i, q in enumerate(all_questions, 1):
            f.write(f"{i}. Question: {q['question']}\n")
            f.write(f"   Answer: {q['gold_answer']}\n")
            if 'faithfulness' in q:
                f.write(f"   Faithfulness: {q['faithfulness']}\n")
            f.write("\n")
    print(f"All questions saved to: {questions_txt_path}")
else:
    print("No questions found to save.")
